#  Racial Bias Detection — Modeling

####  Load Dataset

In [1]:
from datasets import load_dataset

ds = load_dataset("mediabiasgroup/mbib-base")
racial_ds = ds["racial_bias"]

df = racial_ds.to_pandas()

#### Data Cleaning & Text Normalization

In [2]:
import pandas as pd
import numpy as np

# Create a copy for cleaning
clean_df = df.copy()

# Normalize text for duplicate detection
clean_df["normalized_text"] = (
    clean_df["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identify conflicting labels
label_counts = (
    clean_df.groupby("normalized_text")["label"]
    .nunique()
)

conflicting_keys = label_counts[
    label_counts > 1
].index

# Remove conflicting text groups
clean_df = clean_df[
    ~clean_df["normalized_text"].isin(conflicting_keys)
].copy()

# Remove consistent normalized duplicates
clean_df = clean_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Validate dataset
assert clean_df["label"].isin([0, 1]).all()
assert clean_df["normalized_text"].is_unique
assert clean_df["text"].notna().all()

print("Original dataset:", df.shape)
print("Cleaned dataset:", clean_df.shape)

display(clean_df["label"].value_counts())

Original dataset: (9788, 4)
Cleaned dataset: (9784, 5)


label
0    4893
1    4891
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# Repeated Source 75 IDs must stay in the same split.
source75 = clean_df["dataset_id"].eq(75) & clean_df["id"].notna()
clean_df["split_group"] = np.where(
    source75,
    "source75_" + clean_df["id"].astype(str),
    "row_" + clean_df.index.astype(str)
)

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, temp_idx = next(outer.split(
    clean_df["text"], clean_df["label"], groups=clean_df["split_group"]
))
train_df = clean_df.iloc[train_idx].copy()
temp_df = clean_df.iloc[temp_idx].copy()

inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
val_idx, test_idx = next(inner.split(
    temp_df["text"], temp_df["label"], groups=temp_df["split_group"]
))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

for name, subset in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(subset)} rows ({len(subset)/len(clean_df):.1%}) | "
          f"labels: {subset['label'].value_counts(normalize=True).round(3).to_dict()}")


Train: 7763 rows (79.3%) | labels: {0: 0.5, 1: 0.5}
Validation: 1034 rows (10.6%) | labels: {1: 0.503, 0: 0.497}
Test: 987 rows (10.1%) | labels: {0: 0.504, 1: 0.496}


In [4]:
# Verify normalized texts and Source 75 group IDs do not cross splits.
for col in ("normalized_text", "split_group"):
    a,b,c=(set(s[col]) for s in (train_df,val_df,test_df))
    for pair,overlap in (("Train/Validation",a&b),("Train/Test",a&c),("Validation/Test",b&c)):
        print(f"{col} {pair}: {len(overlap)} overlapping")
        assert not overlap, f"{col} leakage in {pair}"
assert len(train_df)+len(val_df)+len(test_df)==len(clean_df)
print("Leakage checks passed.")


normalized_text Train/Validation: 0 overlapping
normalized_text Train/Test: 0 overlapping
normalized_text Validation/Test: 0 overlapping
split_group Train/Validation: 0 overlapping
split_group Train/Test: 0 overlapping
split_group Validation/Test: 0 overlapping
Leakage checks passed.


##  Baseline Models

###  Dummy Classifier and TF-IDF + Logistic Regression

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# 1. Prepare training and validation data
X_train, y_train = train_df["text"], train_df["label"]
X_val, y_val = val_df["text"], val_df["label"]

# 2. Define 3-Fold Cross-Validation strategy
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- A. Dummy Classifier Baseline ---
dummy_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', DummyClassifier(strategy="stratified", random_state=42))
])

dummy_pipeline.fit(X_train, y_train)
dummy_val_pred = dummy_pipeline.predict(X_val)
dummy_val_proba = dummy_pipeline.predict_proba(X_val)[:, 1]

dummy_results = {
    "Model": "Dummy Classifier",
    "ROC-AUC": roc_auc_score(y_val, dummy_val_proba),
    "PR-AUC": average_precision_score(y_val, dummy_val_proba),
    "Accuracy": accuracy_score(y_val, dummy_val_pred),
    "Precision": precision_score(y_val, dummy_val_pred, zero_division=0),
    "Recall": recall_score(y_val, dummy_val_pred, zero_division=0),
    "F1": f1_score(y_val, dummy_val_pred, zero_division=0)
}

# --- B. Logistic Regression Baseline ---
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
lr_val_pred = lr_pipeline.predict(X_val)
lr_val_proba = lr_pipeline.predict_proba(X_val)[:, 1]

lr_results = {
    "Model": "TF-IDF + Logistic Regression",
    "ROC-AUC": roc_auc_score(y_val, lr_val_proba),
    "PR-AUC": average_precision_score(y_val, lr_val_proba),
    "Accuracy": accuracy_score(y_val, lr_val_pred),
    "Precision": precision_score(y_val, lr_val_pred, zero_division=0),
    "Recall": recall_score(y_val, lr_val_pred, zero_division=0),
    "F1": f1_score(y_val, lr_val_pred, zero_division=0)
}

# 3. Combine and Display in a Single Sorted DataFrame Table
baselines_comparison_df = pd.DataFrame([dummy_results, lr_results])
print("=== Baseline Models Comparison Table ===")
display(baselines_comparison_df.round(4))

print("\nDetailed Validation Report for Logistic Regression:")
print(classification_report(y_val, lr_val_pred))

=== Baseline Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.4758,0.4914,0.4758,0.4788,0.4769,0.4778
1,TF-IDF + Logistic Regression,0.7858,0.7892,0.7137,0.7129,0.7212,0.7170



Detailed Validation Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.71      0.71      0.71       514
           1       0.71      0.72      0.72       520

    accuracy                           0.71      1034
   macro avg       0.71      0.71      0.71      1034
weighted avg       0.71      0.71      0.71      1034



##  LSTM Model

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import pandas as pd

# 1. Setup Vocabulary and Sequence Parameters (Fit ONLY on Training data)
VOCAB_SIZE = 10000
MAX_LENGTH = 150

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

# 2. Text to Sequences & Padding
X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq = tokenizer.texts_to_sequences(val_df["text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

# 3. Build Small LSTM Architecture
model_lstm = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')]
)

# 4. Early Stopping Configuration
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 5. Train Model
history = model_lstm.fit(
    X_train_pad, train_df["label"],
    validation_data=(X_val_pad, val_df["label"]),
    batch_size=64,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)

# 6. Evaluate LSTM on Validation Set & Store in Structured Dictionary
val_probs_lstm = model_lstm.predict(X_val_pad).flatten()
val_preds_lstm = (val_probs_lstm >= 0.5).astype(int)

lstm_results = {
    "Model": "Small LSTM",
    "ROC-AUC": roc_auc_score(val_df["label"], val_probs_lstm),
    "PR-AUC": average_precision_score(val_df["label"], val_probs_lstm),
    "Accuracy": accuracy_score(val_df["label"], val_preds_lstm),
    "Precision": precision_score(val_df["label"], val_preds_lstm, zero_division=0),
    "Recall": recall_score(val_df["label"], val_preds_lstm, zero_division=0),
    "F1": f1_score(val_df["label"], val_preds_lstm, zero_division=0)
}

# 7. Display Results in a Clean DataFrame Table
lstm_results_df = pd.DataFrame([lstm_results])
print("\n=== Small LSTM Validation Performance Table ===")
display(lstm_results_df.round(4))

print("\nClassification Report:")
print(classification_report(val_df["label"], val_preds_lstm))

Epoch 1/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.6840 - loss: 0.5922 - roc_auc: 0.7569 - val_accuracy: 0.6412 - val_loss: 0.9301 - val_roc_auc: 0.6581
Epoch 2/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.8220 - loss: 0.4094 - roc_auc: 0.8955 - val_accuracy: 0.6354 - val_loss: 1.4832 - val_roc_auc: 0.6391
Epoch 3/10
122/122 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.8852 - loss: 0.2870 - roc_auc: 0.9499 - val_accuracy: 0.6364 - val_loss: 1.6443 - val_roc_auc: 0.6366
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

=== Small LSTM Validation Performance Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Small LSTM,0.6582,0.6267,0.6412,0.5995,0.8635,0.7076



Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.42      0.54       514
           1       0.60      0.86      0.71       520

    accuracy                           0.64      1034
   macro avg       0.68      0.64      0.62      1034
weighted avg       0.67      0.64      0.62      1034



In [7]:
import pandas as pd

# 1. Combine the results of the three main models into a single DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    lstm_results
])

# 2. Display the final sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

=== Final Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.4758,0.4914,0.4758,0.4788,0.4769,0.4778
1,TF-IDF + Logistic Regression,0.7858,0.7892,0.7137,0.7129,0.7212,0.7170
2,Small LSTM,0.6582,0.6267,0.6412,0.5995,0.8635,0.7076


### Model Comparison — requires rerun after group-aware split

The previous scores shown in the original submission used a split with repeated Source 75 IDs across train/validation/test. Recompute and document validation scores after running this corrected notebook.


In [8]:
import pandas as pd
import numpy as np

# 1. Create an error analysis DataFrame using the Validation Set
error_df = pd.DataFrame({
    "text": val_df["text"],
    "true_label": val_df["label"],
    "predicted_label": val_preds_lstm,
    "bias_probability": val_probs_lstm
})

# Add dataset_id if it exists in your validation data
if "dataset_id" in val_df.columns:
    error_df["dataset_id"] = val_df["dataset_id"]

# 2. Classify error types
error_df["error_type"] = np.select(
    [
        (error_df["true_label"] == 0) & (error_df["predicted_label"] == 1),
        (error_df["true_label"] == 1) & (error_df["predicted_label"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

# 3. Display overall error distribution counts
print("=== Overall Error Distribution ===")
display(error_df["error_type"].value_counts().to_frame("Count"))

# 4. Error distribution by source (if dataset_id exists)
if "dataset_id" in error_df.columns:
    print("\n=== Error Distribution by Source (dataset_id) ===")
    source_errors = pd.crosstab(error_df["dataset_id"], error_df["error_type"])
    display(source_errors)

=== Overall Error Distribution ===


,Count
error_type,
Correct,663
False Positive,300
False Negative,71



=== Error Distribution by Source (dataset_id) ===


error_type,Correct,False Negative,False Positive
dataset_id,,,
75,618,71,300
110,45,0,0


In [9]:
import pickle
from pathlib import Path

# Locate the project root dynamically
project_root = Path.cwd()

if project_root.name == "modeling":
    project_root = project_root.parent.parent
elif project_root.name == "notebooks":
    project_root = project_root.parent

# Define models directory and path
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
tokenizer_path = models_dir / "tokenizer_racial_bias.pkl"
model_path = models_dir / "lstm_racial_bias_model.keras"

# 1. Save Tokenizer configuration
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 2. Save Trained LSTM Model
model_lstm.save(model_path)

print(f"Model saved to: {model_path}")
print(f"Tokenizer saved to: {tokenizer_path}")

Model saved to: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/lstm_racial_bias_model.keras
Tokenizer saved to: /Users/sarahaltaleb/Documents/GitHub/BiasLens/models/tokenizer_racial_bias.pkl


In [10]:
# Verify prediction using the model and tokenizer loaded from disk.
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open(tokenizer_path, "rb") as handle:
    loaded_tokenizer = pickle.load(handle)
loaded_racial_model = tf.keras.models.load_model(model_path, compile=False)

def predict_bias(text, threshold=0.5):
    sequence = loaded_tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=MAX_LENGTH, padding="post", truncating="post")
    probability = float(loaded_racial_model.predict(padded, verbose=0).ravel()[0])
    return {"prediction": "Biased" if probability >= threshold else "Unbiased",
            "bias_probability": round(probability, 4)}

# Reload parity on validation examples, never test examples.
check_text = X_val.iloc[:5].tolist()
check_pad = pad_sequences(loaded_tokenizer.texts_to_sequences(check_text),
                          maxlen=MAX_LENGTH, padding="post", truncating="post")
np.testing.assert_allclose(model_lstm.predict(check_pad, verbose=0),
                           loaded_racial_model.predict(check_pad, verbose=0),
                           rtol=1e-5, atol=1e-5)
print("Model loading test passed.")
print(predict_bias("This is a sample text to test the model."))


Model loading test passed.
{'prediction': 'Biased', 'bias_probability': 0.9183}


In [11]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

cv_strategy = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
X_train = train_df["text"]
y_train = train_df["label"]
cv_groups = train_df["split_group"].to_numpy()

for fold,(train_idx,val_idx) in enumerate(cv_strategy.split(X_train,y_train,groups=cv_groups),start=1):
    overlap=set(cv_groups[train_idx]) & set(cv_groups[val_idx])
    print(f"Fold {fold}: {len(overlap)} overlapping groups")
    assert not overlap, f"Leakage detected in fold {fold}"
print("Group CV check passed.")


Fold 1: 0 overlapping groups
Fold 2: 0 overlapping groups
Fold 3: 0 overlapping groups
Group CV check passed.
